In [1]:
import pandas as pd
import numpy as np
import os

# 1. Load the Raw Macro Data
raw_path = "../data/raw/LKR_Forex_Macro_Raw.csv"
if not os.path.exists(raw_path):
    # Fallback if the path is slightly different
    raw_path = "LKR_Forex_Macro_Raw.csv"

df = pd.read_csv(raw_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Currency', 'Date'])

print(f"Starting Preprocessing for {len(df)} rows...")

# 2. FEATURE ENGINEERING FUNCTION
def create_forex_features(group):
    # --- A. Lags (The Past) ---
    # What was the rate 1 day, 2 days, and 7 days ago?
    for lag in [1, 2, 7]:
        group[f'LKR_Lag_{lag}'] = group['LKR_Rate'].shift(lag)
    
    # --- B. Moving Averages (The Trend) ---
    group['LKR_MA_7'] = group['LKR_Rate'].rolling(window=7).mean()
    group['LKR_MA_30'] = group['LKR_Rate'].rolling(window=30).mean()
    
    # --- C. Volatility (The Risk) ---
    # Daily percentage change
    group['Daily_Return'] = group['LKR_Rate'].pct_change()
    # 7-day rolling standard deviation of returns
    group['LKR_Volatility_7'] = group['Daily_Return'].rolling(window=7).std()
    
    # --- D. Macro-Driver Features ---
    # How much did Gold and Oil change since yesterday?
    group['Gold_Change'] = group['Gold_Price'].pct_change()
    group['Oil_Change'] = group['Oil_Price'].pct_change()
    
    # --- E. THE TARGET (The Future) ---
    # We want to predict the LKR_Rate for the VERY NEXT DAY
    group['Target_Next_Day'] = group['LKR_Rate'].shift(-1)
    
    return group

# Apply the function to each currency group
df = df.groupby('Currency', group_keys=False).apply(create_forex_features)

# 3. ADD CALENDAR & CRISIS FEATURES
# Day of week (0=Mon, 6=Sun)
df['Day_of_Week'] = df['Date'].dt.dayofweek

# Cyclical Month Encoding (Helping the AI understand Jan is close to Dec)
df['Month_Sin'] = np.sin(2 * np.pi * df['Date'].dt.month / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Date'].dt.month / 12)

# Structural Break: 2022 Sri Lankan Economic Crisis
# The LKR was officially floated on March 7, 2022.
df['Is_Crisis_Period'] = (df['Date'] >= '2022-03-07').astype(int)

# 4. FINAL CLEANING
# Drop the NaN rows created by lags/rolling windows/target shift
df_final = df.dropna().copy()

# 5. SAVE PROCESSED DATA
processed_dir = "../data/processed/"
os.makedirs(processed_dir, exist_ok=True)
output_path = os.path.join(processed_dir, "LKR_Forex_Processed.csv")

df_final.to_csv(output_path, index=False)

print(f"\nPREPROCESSING COMPLETE!")
print(f"Final Feature Count: {df_final.shape[1]}")
print(f"Total Training Samples: {len(df_final)}")
print(f"Processed Data Saved to: {output_path}")

Starting Preprocessing for 25218 rows...


C:\Users\menilk\AppData\Local\Temp\ipykernel_31236\1789118962.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('Currency', group_keys=False).apply(create_forex_features)



PREPROCESSING COMPLETE!
Final Feature Count: 20
Total Training Samples: 25038
Processed Data Saved to: ../data/processed/LKR_Forex_Processed.csv
